# Semana 10: Práctica. Rebalanceo, desempeño y VaR con datos reales

**Curso:** Tópicos de Finanzas Avanzadas (ECON-421, UPAO 2026-20)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/10_desempeno_riesgo/clase10_practica.ipynb)

Hoy cerramos la unidad 2 con diez años de datos: simulamos un portafolio con y sin rebalanceo, calculamos todas las medidas de desempeño de nuestros fondos y vemos cómo cambian los rankings, preguntamos cuántos años harían falta para creer en un alfa, y terminamos midiendo el riesgo del portafolio con el VaR por tres métodos y contando cuántas veces los datos lo superaron.

**Requisito previo:** `git pull` en tu fork. Esta semana se agrega a `utils/finanzas.py` la sección "Semana 10": `backtest()`, `medidas_desempeno()`, `drawdown_maximo()`, `var_parametrico()`, `var_historico()`, `var_montecarlo()`, `es_parametrico()`, `es_historico()`, `excepciones_var()` y las medidas escalares (`sharpe`, `treynor`, `jensen`, `m2`, `ratio_informacion`, `ley_fundamental`, `anos_para_significancia`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from utils.french import get_factores
from utils.finanzas import (backtest, medidas_desempeno, drawdown_maximo, anos_para_significancia,
                            var_parametrico, var_historico, var_montecarlo, es_parametrico, es_historico, excepciones_var)

ACCIONES = ["ALICORC1.LM", "FERREYC1.LM", "CPACASC1.LM", "LUSURC1.LM",      # BVL (en soles)
            "BAP", "BVN", "SCCO", "IFS",                                    # Peru en NYSE
            "AAPL", "MSFT", "JNJ", "KO", "XOM", "JPM"]                      # Estados Unidos
FONDOS = {"SPY": "S&P 500", "IWM": "Russell 2000 (pequenas)", "IWN": "Russell 2000 valor",
          "VTV": "Vanguard valor", "VUG": "Vanguard crecimiento", "QQQ": "Nasdaq 100",
          "ARKK": "ARK Innovation", "BRK-B": "Berkshire Hathaway", "EPU": "MSCI Peru"}
MONTO = 100_000                                                             # dolares invertidos

## 1. Los datos: diez años de retornos mensuales en dólares

Mismo tratamiento de las semanas 6, 7 y 9: precios ajustados, acciones de la BVL convertidas a dólares, sin el mes en curso. La tasa libre de riesgo mensual es la de la biblioteca de French (letra del Tesoro a un mes), en la misma moneda y frecuencia que los retornos; el índice de referencia para todos los fondos es SPY.

In [ ]:
bruto = yf.download(ACCIONES + list(FONDOS) + ["PEN=X"], period="10y", interval="1mo",
                    auto_adjust=True, progress=False)["Close"].iloc[:-1]
tc = bruto["PEN=X"].ffill()
precios = bruto.drop(columns="PEN=X")
locales = [t for t in precios.columns if t.endswith(".LM")]
precios[locales] = precios[locales].div(tc, axis=0)

ret = precios.ffill().pct_change(fill_method=None).iloc[1:]
ret.index = ret.index.to_period("M")

try:
    rf = get_factores("ff3")["RF"].reindex(ret.index).ffill()             # mensual, en decimales
    rf = rf.fillna(rf.dropna().iloc[-1])                                  # los ultimos meses aun no publicados: ultimo valor
except Exception as e:
    print("French no respondio, uso 4% anual de respaldo:", e)
    rf = pd.Series(0.04 / 12, index=ret.index)

print(f"Muestra: {ret.index[0]} a {ret.index[-1]} ({len(ret)} meses) | rf promedio anual = {rf.mean() * 12:.2%}")

## 2. *Backtest*: comprar y mantener frente a rebalancear

El portafolio de pesos iguales con las 14 acciones (el humilde $1/n$ que en la semana 7 competía con el tangente). Tres reglas: nunca rebalancear, rebalancear cada mes y rebalancear por bandas (cuando algún peso se aleja más de 2 puntos de su objetivo de 7.1). `backtest()` devuelve el valor del portafolio, los pesos al cierre de cada mes, el número de rebalanceos y la rotación acumulada.

In [ ]:
pesos = np.repeat(1 / len(ACCIONES), len(ACCIONES))
reglas = {"Comprar y mantener": dict(rebalanceo="nunca"),
          "Rebalanceo mensual": dict(rebalanceo="periodico", cada=1),
          "Bandas de 2 puntos": dict(rebalanceo="bandas", banda=0.02)}
bts = {nombre: backtest(ret[ACCIONES], pesos, **kw) for nombre, kw in reglas.items()}

resumen = pd.DataFrame({nombre: {
    "valor final (de 1)": bt["valor"].iloc[-1],
    "retorno anual": bt["valor"].iloc[-1] ** (12 / len(ret)) - 1,
    "volatilidad": bt["valor"].pct_change().std() * np.sqrt(12),
    "caida maxima": drawdown_maximo(bt["valor"]),
    "mayor peso final": bt["pesos"].iloc[-1].max(),
    "rebalanceos": bt["rebalanceos"],
    "rotacion acumulada": bt["rotacion"]} for nombre, bt in bts.items()}).T
print(resumen.round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for nombre, bt in bts.items():
    ax[0].plot(bt["valor"].index.to_timestamp(), bt["valor"], label=nombre)
    ax[1].plot(bt["pesos"].index.to_timestamp(), bt["pesos"].max(axis=1) * 100, label=nombre)
ax[0].set_title("Valor del portafolio (inicio = 1)"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].axhline(100 / 14, color="gray", ls="--", lw=0.8); ax[1].set_title("Mayor peso individual (%)"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Lectura:** mira primero el gráfico de la derecha. Sin rebalancear, el mayor peso individual se aleja del 7.1 inicial: el portafolio se va concentrando en las acciones que más subieron, y al final tiene la volatilidad y la caída máxima que ese portafolio concentrado le impone. Rebalancear cada mes mantiene los pesos pero opera siempre; las bandas operan solo cuando hace falta. ¿Cuál de las tres reglas rindió más? Depende de la muestra: si las ganadoras siguieron ganando (tendencia), comprar y mantener gana; si alternaron (reversión), rebalancear gana (es la prima de rebalanceo). Lo que no depende de la muestra es el control del riesgo. ¿Cuántas veces operaron las bandas frente al rebalanceo mensual, y cuánta rotación acumularon? Esa diferencia, multiplicada por el costo de transacción, es lo que las bandas ahorran.

## 3. Todas las medidas de desempeño de nuestros fondos

Una sola función devuelve, para cada fondo, retorno y volatilidad anualizados, beta y alfa de Jensen (con su $t$), Sharpe, Treynor, $M^2$, *tracking error*, ratio de información, $R^2$ y caída máxima, todo frente a SPY y con la $R_f$ mensual de French. También evaluamos los tres portafolios del *backtest* como si fueran fondos.

In [ ]:
ret_bt = pd.DataFrame({nombre: bt["valor"].pct_change() for nombre, bt in bts.items()}).iloc[1:]
candidatos = pd.concat([ret[list(FONDOS)], ret_bt], axis=1)
tabla = medidas_desempeno(candidatos, rf, ret["SPY"])
cols = ["retorno", "sigma", "beta", "alfa", "t_alfa", "sharpe", "treynor", "m2", "tracking_error", "ir", "r2", "caida_maxima"]
tabla[cols].round(3).sort_values("sharpe", ascending=False)

In [ ]:
# Dos rankings y por que difieren
rk = pd.DataFrame({"por Sharpe": tabla["sharpe"].rank(ascending=False),
                   "por Treynor": tabla["treynor"].rank(ascending=False),
                   "por alfa": tabla["alfa"].rank(ascending=False),
                   "por IR": tabla["ir"].rank(ascending=False, na_option="bottom"),     # SPY contra si mismo: TE = 0, IR indefinido
                   "R2": tabla["r2"].round(2)}).sort_values("por Sharpe")
print(rk.to_string())

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(tabla["sharpe"], tabla["treynor"], s=40, color="#000080")
for t in tabla.index:
    ax.annotate(t, (tabla.loc[t, "sharpe"], tabla.loc[t, "treynor"]), xytext=(5, 3), textcoords="offset points", fontsize=8)
ax.set_xlabel("Sharpe (riesgo total)"); ax.set_ylabel("Treynor (riesgo sistematico)")
ax.set_title("Dos medidas, dos rankings"); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

**Para discutir:** ¿qué fondos cambian más de posición entre el ranking por Sharpe y el ranking por Treynor? Busca los de $R^2$ bajo (Berkshire, EPU, los portafolios de acciones peruanas): su riesgo idiosincrático castiga el Sharpe y no el Treynor, y además hace que su beta esté mal estimado. Mira el $M^2$: es el Sharpe traducido a puntos de retorno anual sobre SPY, el número que se le dice a un cliente. Y mira el ratio de información: los fondos indexados a SPY tienen *tracking error* casi nulo e IR sin sentido; los demás, un IR que casi nunca llega a 0.5. ¿Cuál es el IR de QQQ y qué dice de la década?

## 4. ¿Habilidad o suerte?

Cada alfa viene con su estadístico $t$. Con el $\sigma_\varepsilon$ implícito en la regresión, calculamos cuántos años de historial harían falta para que ese alfa fuera significativo si se mantuviera.

In [ ]:
t2 = tabla.drop("SPY")                                                               # SPY es el indice: alfa cero por construccion
sigma_eps = (t2["sigma"] ** 2 * (1 - t2["r2"])).clip(lower=0) ** 0.5                 # volatilidad idiosincratica anual
suerte = pd.DataFrame({"alfa anual": t2["alfa"], "t": t2["t_alfa"], "sigma_eps": sigma_eps,
                       "anos observados": t2["n"] / 12,
                       "anos para t = 2": [anos_para_significancia(a, s) for a, s in zip(t2["alfa"], sigma_eps)]})
print(suerte.round(2).sort_values("t", ascending=False).to_string())
print("\nFondos con |t| > 2:", list(suerte.index[suerte["t"].abs() > 2]))

**Lectura:** ¿algún alfa supera $|t| = 2$? Casi nunca con diez años (SPY queda fuera de la tabla: es el índice y su alfa es cero por construcción). La columna final dice cuántos años harían falta: decenas o cientos para la mayoría. Es la misma conclusión de ARKK en la semana 9, generalizada: con la historia disponible no se puede distinguir habilidad de suerte, y por eso la evaluación profesional de un gestor mira el proceso (consistencia del estilo, riesgo activo, costos) tanto como el resultado.

## 5. El VaR del portafolio por tres métodos

Tomamos el portafolio rebalanceado por bandas (el que recomendaríamos) y calculamos su VaR a un mes, al 95 y al 99 por ciento, sobre 100,000 dólares: paramétrico (normal con la media y la volatilidad mensuales), histórico (el percentil de los 10 años de retornos mensuales) y Monte Carlo (normal simulada y *bootstrap* de los retornos observados). Y el *expected shortfall* por los dos primeros.

In [ ]:
r_p = ret_bt["Bandas de 2 puntos"].dropna()
mu_m, sig_m = r_p.mean(), r_p.std()
print(f"Portafolio con bandas: retorno mensual medio {mu_m:.2%}, volatilidad mensual {sig_m:.2%} ({len(r_p)} meses)\n")

filas = {}
for nivel in (0.95, 0.99):
    filas[f"VaR {nivel:.0%}"] = {
        "parametrico": var_parametrico(mu_m, sig_m, nivel, MONTO),
        "historico": var_historico(r_p, nivel, MONTO),
        "Monte Carlo normal": var_montecarlo(mu_m, sig_m, nivel, MONTO, semilla=10),
        "Monte Carlo bootstrap": var_montecarlo(mu_m, sig_m, nivel, MONTO, semilla=10, retornos=r_p)}
    filas[f"ES {nivel:.0%}"] = {"parametrico": es_parametrico(mu_m, sig_m, nivel, MONTO), "historico": es_historico(r_p, nivel, MONTO)}
var_tabla = pd.DataFrame(filas).T
print(var_tabla.round(0).to_string())

from scipy.stats import norm, skew, kurtosis
print(f"\nAsimetria = {skew(r_p):.2f} | curtosis en exceso = {kurtosis(r_p):.2f} (la normal tiene 0 y 0)")
print(f"Peor mes de la muestra: {r_p.idxmin()} con {r_p.min():.1%} = {-r_p.min() * MONTO:,.0f} dolares")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(r_p * 100, bins=30, color="#000080", alpha=0.6, density=True)
x = np.linspace(r_p.min() * 100, r_p.max() * 100, 200)
ax.plot(x, norm.pdf(x, mu_m * 100, sig_m * 100), color="#E36C0A", lw=2, label="normal ajustada")
for nombre, color in [("parametrico", "#E36C0A"), ("historico", "black")]:
    ax.axvline(-var_tabla.loc["VaR 95%", nombre] / MONTO * 100, color=color, ls="--", label=f"VaR 95% {nombre}")
ax.set_xlabel("Retorno mensual (%)"); ax.set_title("Distribucion de los retornos mensuales y VaR a un mes"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Lectura:** el Monte Carlo normal reproduce el paramétrico (misma distribución, más ruido de simulación), y el *bootstrap* reproduce el histórico: cada método es tan bueno como su supuesto. Compara el histórico con el paramétrico: con colas gordas (curtosis en exceso positiva) el histórico tiende a salir mayor, pero con 117 meses el percentil 99 histórico es una interpolación entre los dos peores meses y es muy ruidoso; el ES histórico, que promedia la cola, sí sale claramente por encima del paramétrico. El ES responde la pregunta que el VaR deja abierta: cuánto se pierde, en promedio, en el 5 por ciento de los meses malos. Y el peor mes de la muestra (busca marzo de 2020) muestra que la pérdida realizada puede ser muy superior al VaR.

## 6. *Backtest* del VaR: contar las excepciones

Un VaR al 95 por ciento bien calibrado debe ser superado en el 5 por ciento de los meses. Lo calculamos de forma rodante (con los 36 meses anteriores, sin mirar el futuro) y contamos cuántas veces la pérdida del mes lo superó.

In [ ]:
VENTANA = 36
var_rod = pd.Series({t: var_parametrico(r_p.iloc[i - VENTANA:i].mean(), r_p.iloc[i - VENTANA:i].std(), 0.95)
                     for i, t in enumerate(r_p.index) if i >= VENTANA})
var_hist_rod = pd.Series({t: var_historico(r_p.iloc[i - VENTANA:i], 0.95) for i, t in enumerate(r_p.index) if i >= VENTANA})
prueba = r_p.loc[var_rod.index]

for nombre, serie in [("parametrico", var_rod), ("historico", var_hist_rod)]:
    e = excepciones_var(prueba, serie)
    print(f"VaR 95% {nombre:11s} rodante ({VENTANA} meses): {e['excepciones']} excepciones en {e['periodos']} meses = {e['fraccion']:.1%} (esperado 5%)")

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.bar(prueba.index.to_timestamp(), prueba * 100, width=25, color=np.where(prueba < -var_rod, "red", "#000080"), alpha=0.7)
ax.plot(var_rod.index.to_timestamp(), -var_rod * 100, color="#E36C0A", lw=2, label="VaR 95% parametrico rodante")
ax.plot(var_hist_rod.index.to_timestamp(), -var_hist_rod * 100, color="black", lw=1.2, ls="--", label="VaR 95% historico rodante")
ax.set_ylabel("Retorno mensual (%)"); ax.set_title("Backtest del VaR: en rojo, los meses que lo superaron"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Lectura:** ¿la fracción de excepciones está cerca del 5 por ciento? Con 80 o 90 meses de prueba se esperan 4 o 5 excepciones; más de 8 indica que el VaR subestima el riesgo. Fíjate dónde se agrupan las excepciones: en las crisis, una tras otra, cuando el VaR calibrado en meses tranquilos todavía no reaccionó. Esa agrupación es el argumento para complementar el VaR con escenarios de estrés y con límites de pérdida, y es lo que la unidad 3 retoma desde la macroeconomía.

## 7. Cierre

Desde hoy quedan en la librería, en la sección "Semana 10" de `utils/finanzas.py`: `backtest()`, `medidas_desempeno()`, `drawdown_maximo()`, los VaR por tres métodos con sus *expected shortfall*, `excepciones_var()` y las medidas escalares de la teoría.

**Ejercicio opcional** (`clase10_tarea.ipynb`, sin entrega ni nota): el *backtest* con rebalanceo de tu universo de seis activos, las medidas de desempeño de tu empresa y de tu fondo frente al índice, los años que harían falta para creer en su alfa, y el VaR de tu portafolio por tres métodos con su *backtest*. Esta semana no hay tarea semanal.

**Próxima semana (miércoles 18/11):** PC2, informe país y exposición en parejas (`examenes/pc2_consigna.pdf`), y teoría de las semanas 11 y 12: factores macroeconómicos y estructura temporal de tasas.